<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.2-heat-and-wave/Ex07.2_03_panel_both_ics_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_07.2 · Notebook 03 — The Struck Panel

**Paired with L7.2 · Fundamental PDEs**

A tensioned panel, clamped on four edges, struck at $t = 0$. It starts **flat**
and **moving**:

$$u_{tt} = c^2(u_{xx} + u_{yy}), \qquad u|_{\partial\Omega} = 0$$
$$u(x,y,0) = 0, \qquad u_t(x,y,0) = v_0 \sin(\pi X)\sin(\pi Y)$$

Second order in time, so **two** initial conditions. The die needed one. That
difference is not bookkeeping, and notebook 04 makes it cost something.

## What you will do

1. Write the hyperbolic residual — note it needs `d2` in time, not `grad`.
2. Supply both initial conditions and solve the panel properly.
3. Check the answer against the exact solution over three periods.
4. Watch the error grow with time, and say why it must.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.2-heat-and-wave/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · The residual, and the second initial condition

$\mathcal{F} = \hat u_{,tt} - c^2(\hat u_{,xx} + \hat u_{,yy})$.

Time is column 2, so the second time derivative is `d2(u, xyt, 2)`. The
velocity condition needs a *first* time derivative on the initial slice:
`grad(u, xyt)[:, 2:3]` evaluated at $t = 0$.

### Your turn

In [ ]:
# TODO 1 --- the wave residual and the four-term loss ----------------------------------------------------
# Three `...` to replace:
#   line 1  ->  u_tt - pb.C_WAVE ** 2 * lap                   u_tt = c^2 (u_xx + u_yy)
#   line 2  ->  grad(u_pred, xyt_0)[:, 2:3]                   the velocity at t = 0: differentiate through xyt_0
#   line 3  ->  mse(f) + w_b*mse(b) + w_u*mse(iu) + w_v*mse(iv)
def wave_residual(model, xyt):
    u = model(xyt)
    u_tt = d2(u, xyt, 2)
    lap  = d2(u, xyt, 0) + d2(u, xyt, 1)
    return ...                                    # <- u_tt - pb.C_WAVE ** 2 * lap

U_SCALE = pb.V_STRIKE / pb.OMEGA               # metres
R_SCALE = U_SCALE * pb.OMEGA ** 2              # m/s^2

def make_loss(model, xyt_f, xyt_b, xyt_0, u0, v0, w_b=1.0, w_u=1.0, w_v=1.0):
    def loss():
        f = wave_residual(model, xyt_f) / R_SCALE
        b = model(xyt_b) / U_SCALE
        u_pred = model(xyt_0)
        v_pred = ...                              # <- grad(u_pred, xyt_0)[:, 2:3]
        iu = (u_pred - u0) / U_SCALE
        iv = (v_pred - v0) / pb.V_STRIKE
        return ...                                # <- mse(f) + w_b*mse(b) + w_u*mse(iu) + w_v*mse(iv)
    return loss
# ------------------------------------------------------------------------------

## 2 · Train

A wave problem is harder than a diffusion problem for a PINN, and it is worth
knowing why before you watch it struggle. Diffusion *forgets*: an error
introduced early decays away. A wave **propagates** — an error made at $t=0$
travels, reflects off the edges and is still there three periods later. There
is nothing in the physics that removes it.

In [ ]:
N_F, N_B, N_0 = 6000, 30, 800
T_SPAN = (0.0, pb.WAVE_T_END)

set_seed(88)
model = MLP(n_in=3, n_hidden=48, n_layers=5)
describe(model, N_F)

xyt_f = to_tensor(spacetime_points(N_F, pb.WAVE_DOMAIN, T_SPAN, seed=1),
                  requires_grad=True)
xyt_b = to_tensor(boundary_points_in_time(N_B, 24, pb.WAVE_DOMAIN, T_SPAN, seed=1))

init_np = initial_points(N_0, pb.WAVE_DOMAIN, t0=0.0, seed=1)
xyt_0 = to_tensor(init_np, requires_grad=True)
u0 = to_tensor(pb.wave_exact(init_np[:, 0], init_np[:, 1], 0.0).reshape(-1, 1))
v0 = to_tensor(pb.wave_velocity(init_np[:, 0], init_np[:, 1], 0.0).reshape(-1, 1))

print(f"\ninitial displacement, max |.| = {float(u0.abs().max()):.2e} m   <- flat")
print(f"initial velocity,     max |.| = {float(v0.abs().max()):.4f} m/s <- moving")

history = train_two_stage(model, make_loss(model, xyt_f, xyt_b, xyt_0, u0, v0),
                          adam_steps=6000, lbfgs_steps=250, lr=1e-3)
plot_curves(history, title="the panel — both initial conditions supplied")
plt.show()

## 3 · Does it oscillate?

The first question for a wave solution is not "how small is the error" but
"does it have the right period". A model can have a respectable norm and be
slowly drifting out of phase, which is a different and worse failure.

### Your turn

In [ ]:
# TODO 2 --- the panel centre through time, and the period --------------------------------------------------
# Two `...` to replace:
#   line 1  ->  pb.wave_exact(q[:, 0], q[:, 1], ts)                          the exact centre history
#   line 2  ->  np.where(np.sign(centre_pred[1:]) != np.sign(centre_pred[:-1]))[0]   sign changes = zero crossings
ts = np.linspace(0.0, pb.WAVE_T_END, 400)
mid = pb.L_PANEL / 2
q = np.stack([np.full_like(ts, mid), np.full_like(ts, mid), ts], axis=1)
with torch.no_grad():
    centre_pred = to_numpy(model(to_tensor(q))).ravel()
centre_exact = ...                                # <- pb.wave_exact(q[:, 0], q[:, 1], ts)

crossings = ...                                   # <- np.where(np.sign(centre_pred[1:]) != np.sign(centre_pred[:-1]))[0]
t_cross = ts[crossings]
period_pred = 2.0 * float(np.mean(np.diff(t_cross)))   # half a period between consecutive crossings
# ------------------------------------------------------------------------------

In [ ]:
pb.plot_time_history(ts, {"model": centre_pred * 1e3,
                          "exact": centre_exact * 1e3},
                     title="Panel centre — three periods",
                     ylabel="displacement [mm]")
plt.show()

print(f"  exact period    : {pb.wave_period()*1e3:.3f} ms")
print(f"  measured period : {period_pred*1e3:.3f} ms"
      f"   ({100*(period_pred/pb.wave_period()-1):+.2f}%)")
print(f"  peak, exact     : {np.abs(centre_exact).max()*1e3:.4f} mm")
print(f"  peak, model     : {np.abs(centre_pred).max()*1e3:.4f} mm")

**What you should see.** Two curves that track each other for the first period
or two and then begin to separate — usually in amplitude first, phase later.

A period within a percent or so is a good result. If yours has drifted more
than that, the model is solving a wave equation with the wrong speed, which no
amount of pointwise error reporting would have told you.

---

## 4 · Error against time

In [ ]:
TIMES = np.linspace(0.0, pb.WAVE_T_END, 9)
X, Y, pts = grid_points(101, 101, pb.WAVE_DOMAIN)

rel, mx = [], []
for t in TIMES:
    q = np.concatenate([pts, np.full((len(pts), 1), t)], axis=1)
    with torch.no_grad():
        pred = to_numpy(model(to_tensor(q))).ravel()
    ref = pb.wave_exact(pts[:, 0], pts[:, 1], t)
    if np.abs(ref).max() < 1e-12:        # t = 0: the panel is flat
        rel.append(np.nan); mx.append(np.abs(pred).max())
    else:
        rel.append(relative_l2(pred, ref)); mx.append(max_abs_error(pred, ref))

print(error_table([[f"{t*1e3:.1f}", "n/a" if np.isnan(r) else f"{r:.3e}",
                    f"{m*1e6:.2f}"] for t, r, m in zip(TIMES, rel, mx)],
                  ["t [ms]", "relative L2", "worst error [µm]"]))

fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.plot(TIMES*1e3, np.array(mx)*1e6, "o-", lw=1.9, ms=6, color="#d94f2b")
ax.set_xlabel("t [ms]"); ax.set_ylabel("worst error [µm]")
ax.set_title("A wave does not forget its mistakes")
ax.grid(alpha=0.25)
plt.show()

**What you should see.** Error that **grows** with time, roughly steadily.

Contrast that with notebook 01, where the die's error *fell* with time. The
mechanism is the difference between the two equations. Diffusion is
dissipative — it destroys information, including the information in your
errors. The wave equation is conservative: whatever you get wrong is carried
along and reflected, forever.

Note also the `n/a` at $t = 0$. The exact field there is identically zero, so a
relative norm has nothing to divide by. That is why the table reports the
absolute error in micrometres beside it — and why `relative_l2` in `pinn_core`
refuses a zero reference rather than returning a misleading number.

---

## 5 · Save

In [ ]:
os.makedirs("Ex07.2_outputs", exist_ok=True)
path = os.path.join("Ex07.2_outputs", "nb03_panel.npz")
np.savez(path, times=TIMES, rel=np.asarray(rel, dtype=float),
         max_err=np.asarray(mx, dtype=float),
         period_pred=period_pred, period_exact=pb.wave_period(),
         centre_pred=centre_pred, centre_exact=centre_exact, ts=ts,
         adam=history["adam"], lbfgs=history["lbfgs"])
torch.save(model.state_dict(), os.path.join("Ex07.2_outputs", "nb03_panel.pt"))
print("wrote", path)

## 6 · Before you move on

1. The die's error fell with time and the panel's grew. Give the mechanism.
2. Why does a relative $L^2$ norm fail at $t = 0$ here but not in notebook 01?
3. The period is a better first check than the pointwise error. Explain what
   failure it catches that a norm would miss.
4. You supplied both initial conditions. Before turning the page: what do you
   think happens if you supply only the displacement?

Next: **notebook 04**, which answers question 4 by doing it.